In [1]:
import earthaccess
import xarray as xr
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from datetime import datetime, timedelta
import random

c:\Users\xmk0904\PycharmProjects\WeatherBenders\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### NASA Data Access

In [ ]:
auth = earthaccess.login()

In [ ]:
token = requests.get(signin_url, auth=HTTPBasicAuth(netrc.netrc().hosts['urs.earthdata.nasa.gov'][0], 
                                                    netrc.netrc().hosts['urs.earthdata.nasa.gov'][2]),
                     allow_redirects=True).text.replace('"','')

In [ ]:
'''def query(product="GPM_3IMERGDF", lat_bounds, lon_bounds, start_date, end_date)
    try:    
        results = earthaccess.search_data(
            short_name=product,
            temporal=(start_date, end_date),
            bounding_box=(lon_bounds[0], lat_bounds[0], lon_bounds[1], lat_bounds[1])
        )
        print(f"Found {len(results)} files")
    except Exception as e
        print(f"Couldn't query the data due to {e} exception")
    try:    
        files = earthaccess.download(results, "./gpm_data")
        print("Downloaded query results in ./gpm_data")
        ds = xr.open_mfdataset(files, engine="netcdf4")
        return ds
    except Exception as e
        print(f"Couldn't download the data due to {e} exception")'''

In [ ]:
def call_time_series(lat, lon, time_start, time_end, data="GPM_3IMERGDF"):
    """
    INPUTS:
    lat - latitude
    lon - longitude
    time_start - start of time series in YYYY-MM-DDThh:mm:ss format (UTC)
    end_time - end of the time series in YYYY-MM-DDThh:mm:ss format (UTC)
    data - name of the data parameter for the time series
    
    OUTPUT:
    time series csv output string
    """
    query_parameters = {
        "data":data,
        "location":"[{},{}]".format(lat,lon),
        "time":"{}/{}".format(time_start,time_end)
    }
    headers = {"authorizationtoken":token}
    response=requests.get(time_series_url,params=query_parameters,headers=headers)
    return response.text

In [ ]:
def parse_csv(ts):
    """
    INPUTS:
    ts - time series output of the time series service
    
    OUTPUTS:
    headers,df - the headers from the CSV as a dict and the values in a pandas dataframe
    """
    with io.StringIO(ts) as f:
        # the first 13 rows are header
        headers = {}
        for i in range(13):
            line = f.readline()
            key,value = line.split(",")
            headers[key] = value.strip()

        # Read the csv proper
        df = pandas.read_csv(
            f,
            header=1,
            names=("Timestamp",headers["param_name"]),
            converters={"Timestamp":pandas.Timestamp}
        )

    return headers, df

In [ ]:
'''def data_to_csv(ds, ds_vars)
    try:
        for var in ds_vars:
            var = ds[var].mean(dim=["lon", "lat"])
            var_series = precip.to_pandas()
            var_series.name = var
            precip_series = precip_series.dropna()'''


In [ ]:
data = "GPM_3IMERGDF"  # GPM IMERG Final Daily
lat = 19.4263  # Approximate CDMX
lon = -99.1266
start_date = "2015-01-01"
end_date = "2025-01-01"

In [ ]:
ts = call_time_series(lat,lon,start_date,end_date,data)

In [ ]:
headers,df = parse_csv(ts)
print(df.head())

### Fake Data generation for testing

In [15]:
# Define start and end dates
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 12, 31)

# Generate a list of daily timestamps between the two dates
date_range = [start_date + timedelta(days=i) for i in range((end_date - start_date).days + 1)]

# Number of samples (same as number of dates)
n = len(date_range)

# Generate fake data
data = {
    "Timestamp": [d.strftime("%Y-%m-%d") for d in date_range],
    "precipitation_mm": [round(60 * random.betavariate(1, 5), 2) for _ in range(n)],
    "temperature_cent": [round(random.uniform(0.0, 39.0), 1) for _ in range(n)],
    "wind_m/s": [round(10 * random.betavariate(1, 5), 2) for _ in range(n)]
}

# Create DataFrame
df = pd.DataFrame(data)
print(df.head())

    Timestamp  precipitation_mm  temperature_cent  wind_m/s
0  2024-01-01              3.00               3.8      0.34
1  2024-01-02              2.45              26.3      2.85
2  2024-01-03              5.68               9.7      0.73
3  2024-01-04              6.70               2.2      0.01
4  2024-01-05             18.32               2.4      2.58


### Model training

In [16]:
# --- FEATURE ENGINEERING ---
df = df.sort_values("Timestamp").reset_index(drop=True)

print(df.head())
print(df.describe())

    Timestamp  precipitation_mm  temperature_cent  wind_m/s
0  2024-01-01              3.00               3.8      0.34
1  2024-01-02              2.45              26.3      2.85
2  2024-01-03              5.68               9.7      0.73
3  2024-01-04              6.70               2.2      0.01
4  2024-01-05             18.32               2.4      2.58
       precipitation_mm  temperature_cent    wind_m/s
count        366.000000        366.000000  366.000000
mean           9.903361         19.108743    1.628825
std            8.515689         10.985796    1.433709
min            0.050000          0.100000    0.000000
25%            3.235000         10.300000    0.542500
50%            7.460000         18.700000    1.230000
75%           14.087500         27.775000    2.277500
max           42.770000         38.900000    8.330000


In [17]:
def classify_weather(row):
    precip = row["precipitation_mm"]
    temp = row["temperature_cent"]
    wind = row["wind_m/s"]
    
    # Ideal: mild temp, low wind, minimal rain
    if (precip < 2) and (18 <= temp <= 28) and (wind < 5):
        return "ideal"
    
    # Extreme: heavy rain, strong wind, or very high/low temp
    elif (precip >= 20) or (temp <= 5) or (temp >= 38) or (wind >= 12):
        return "extreme"
    
    # Non-ideal: everything else (e.g., light rain, moderate wind, etc.)
    else:
        return "non-ideal"

In [18]:
df["category"] = df.apply(classify_weather, axis=1)
print(df["category"].value_counts())

category
non-ideal    250
extreme       98
ideal         18
Name: count, dtype: int64


In [20]:
X = df[["precipitation_mm", "temperature_cent", "wind_m/s"]]
y = df["category"].map({"ideal": 0, "non-ideal": 1, "extreme": 2})

In [21]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [23]:
split_ratio = 0.8
split_index = int(len(df) * split_ratio)
X_train, X_test = X_scaled[:split_index], X_scaled[split_index:]
y_train, y_test = y[:split_index], y[split_index:]
dates_test = df["Timestamp"].iloc[split_index:]

In [24]:
clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    class_weight="balanced"
)
clf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10, n_estimators=300,
                       random_state=42)

In [25]:
y_pred = clf.predict(X_test)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=["ideal", "non-ideal", "extreme"]))


Classification Report:

              precision    recall  f1-score   support

       ideal       1.00      1.00      1.00         3
   non-ideal       1.00      1.00      1.00        55
     extreme       1.00      1.00      1.00        16

    accuracy                           1.00        74
   macro avg       1.00      1.00      1.00        74
weighted avg       1.00      1.00      1.00        74



In [34]:
def predict_probability(n):
    future_X = X_scaled[-n:]
    future_dates = df["Timestamp"].iloc[-n:]
    future_probs = clf.predict_proba(future_X)

    prob_df = pd.DataFrame(
        future_probs,
        columns=["P(ideal)", "P(non-ideal)", "P(extreme)"],
        index=future_dates
    )
    print(f"\nPredicted Weather Probabilities (next {n} days):")
    print(prob_df)

In [35]:
predict_probability(30)


Predicted Weather Probabilities (next 30 days):
            P(ideal)  P(non-ideal)  P(extreme)
Timestamp                                     
2024-12-02  0.000657      0.985230    0.014113
2024-12-03  0.016667      0.959444    0.023889
2024-12-04  0.000000      0.985603    0.014397
2024-12-05  0.000000      0.829919    0.170081
2024-12-06  0.000000      0.391807    0.608193
2024-12-07  0.000657      0.934515    0.064829
2024-12-08  0.000000      0.172267    0.827733
2024-12-09  0.000657      0.984393    0.014950
2024-12-10  0.000000      0.050317    0.949683
2024-12-11  0.000657      0.993529    0.005814
2024-12-12  0.000000      0.115601    0.884399
2024-12-13  0.000000      0.125986    0.874014
2024-12-14  0.000000      0.975387    0.024613
2024-12-15  0.000000      0.893632    0.106368
2024-12-16  0.003333      0.990695    0.005971
2024-12-17  0.000657      0.968555    0.030788
2024-12-18  0.000000      0.917224    0.082776
2024-12-19  0.087323      0.862677    0.050000
2024-12-20 